In [ ]:
import numpy as np
import pandas as pd
import sys
import ast

sys.path.append('.')
from utils import to_latex_table

# Load y_train
train_df = pd.read_csv("../../data/vigoemotions/train.csv")
num_labels = 28
y_train = np.zeros((len(train_df), num_labels), dtype=np.float32)

for i, raw in enumerate(train_df['labels']):
    indices = ast.literal_eval(str(raw)) if isinstance(raw, str) else [int(raw)]
    if isinstance(indices, int): indices = [indices]
    for idx in indices:
        if 0 <= idx < num_labels: y_train[i, idx] = 1.0


In [ ]:
# 6 emotion clusters from TACO paper mapping for ViGoEmotions
# M is 6 x 28
M = np.zeros((6, 28))
# Cluster 1 (Anger/Disgust): 1(anger), 7(disgust), 12(annoyance), 13(disapproval)
M[0, [1, 7, 12, 13]] = 1
# Cluster 2 (Fear/Sadness): 14(fear), 15(nervousness), 23(sadness), 24(disappointment), 25(grief), 26(remorse)
M[1, [14, 15, 23, 24, 25, 26]] = 1
# Cluster 3 (Joy/Amusement): 16(joy), 17(amusement), 18(approval), 19(caring), 20(excitement), 21(gratitude), 22(love), 27(optimism), 28(relief)->27
# (simplified)
M[2, [16, 17, 18, 19, 20, 21, 22, 27]] = 1
# Cluster 4 (Surprise): 2(anticipation), 3(surprise), 4(realization), 5(confusion), 6(curiosity)
M[3, [2, 3, 4, 5, 6]] = 1
# Cluster 5 (Trust): 8(trust), 9(admiration), 10(pride), 11(desire)
M[4, [8, 9, 10, 11]] = 1
# Cluster 6 (Neutral): 0(neutral)
M[5, [0]] = 1

batch_size = 32
num_batches = len(y_train) // batch_size

false_negatives_1 = 0
false_negatives_2 = 0
total_negative_pairs = 0

for b in range(num_batches):
    start = b * batch_size
    end = start + batch_size
    batch_y = y_train[start:end]
    
    # Compute s_bin using Eq 14: y_i * M^T * M * y_j^T > 0
    s_bin = (batch_y @ M.T @ M @ batch_y.T) > 0
    
    for i in range(batch_size):
        for j in range(i+1, batch_size):
            is_pos = s_bin[i, j]
            shared_labels = np.sum(batch_y[i] * batch_y[j])
            
            if not is_pos:
                total_negative_pairs += 1
                if shared_labels >= 1:
                    false_negatives_1 += 1
                if shared_labels >= 2:
                    false_negatives_2 += 1

results = [{
    "Total Negative Pairs": total_negative_pairs,
    "False Negatives (≥1 shared label)": f"{false_negatives_1} ({(false_negatives_1/total_negative_pairs)*100:.2f}%)",
    "False Negatives (≥2 shared labels)": f"{false_negatives_2} ({(false_negatives_2/total_negative_pairs)*100:.2f}%)"
}]

df = pd.DataFrame(results)
print(to_latex_table(df, "TACO Cluster Pairing Audit", "tab:taco_audit"))
